# Notebook Overview

## Gold Sales Fact (`gold_fact_sales`)


### Purpose
Build the Gold sales fact table by enriching Silver transaction records with customer, product, order, and calendar information. This table serves as the central analytical dataset for sales reporting and business intelligence.

### Source Tables
- Silver Transactions: `silver/transactions`
- Silver Orders: `silver/orders`
- Gold Customer Dimension: `gold/dim_customer`
- Gold Product Dimension: `gold/dim_product`
- Gold Date Dimension: `gold/dim_date`

### Transformation Logic
1. Load Silver transactions and orders.
2. Load the three previously created Gold dimensions.
3. Join transactions to orders using `order_id`.
4. Enrich transactions with customer attributes through `customer_id`.
5. Enrich transactions with product attributes through `product_id`.
6. Derive and attach `date_key` using the transaction date.
7. Select transaction-level identifiers, dimension keys, order attributes, and financial measures.
8. Add Gold processing metadata.
9. Validate the fact table's grain, row count, and referential matching.
10. Write the result to the Gold layer and verify the persisted output.

### Target
- **Gold fact:** `gold_fact_sales`
- **Output path:** `gold/fact_sales`
- **Storage format:** Parquet
- **Grain:** One row per `transaction_id`

### Measures
- `quantity`
- `unit_price`
- `discount_amount`
- `tax_amount`
- `transaction_amount`

### Dimension Keys
- `customer_id`
- `product_id`
- `date_key`

### Data Quality Checks
- Preserve the Silver transaction row count.
- Verify transaction IDs remain unique and non-null.
- Check for row multiplication caused by joins.
- Measure transactions without matching orders, customers, products, or dates.
- Confirm required financial measures retain their source values.
- Read back the saved Gold fact and verify its row count.

### Design Notes
- All joins use LEFT JOINs to preserve the original transaction population.
- Missing dimension relationships are retained as nulls and reported in quality metrics.
- The fact table is transaction-grain, not order-grain.
- Order-level measures such as `order_total` are intentionally excluded to avoid double-counting when an order contains multiple transactions.
- Silver data is read-only; all Gold output is written to a separate path.

### Configuration / Paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

FILE_PATH = "/content/drive/MyDrive/pyspark-ecommerce-analytics-data"

SILVER_TRANSACTION_PATH = f"{FILE_PATH}/silver/transactions"
SILVER_ORDER_PATH = f"{FILE_PATH}/silver/orders"

GOLD_CUSTOMER_PATH = f"{FILE_PATH}/gold/dim_customer"
GOLD_PRODUCT_PATH = f"{FILE_PATH}/gold/dim_product"
GOLD_DATE_PATH = f"{FILE_PATH}/gold/dim_date"

GOLD_FACT_SALES_PATH = f"{FILE_PATH}/gold/fact_sales"

EXPECTED_TRANSACTION_COUNT = 49990

Mounted at /content/drive


### Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("gold-fact-sales")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.0.5


### Load Silver Source Tables

In [ ]:
df_transaction = (
    spark.read
    .format("parquet")
    .load(SILVER_TRANSACTION_PATH)
)

df_order = (
    spark.read
    .format("parquet")
    .load(SILVER_ORDER_PATH)
)

print("Silver transaction count:", df_transaction.count())
print("Silver order count:", df_order.count())

Silver transaction count: 49990
Silver order count: 29996


### Load Gold Dimension Tables

In [ ]:
df_customer = (
    spark.read
    .format("parquet")
    .load(GOLD_CUSTOMER_PATH)
)

df_product = (
    spark.read
    .format("parquet")
    .load(GOLD_PRODUCT_PATH)
)

df_date = (
    spark.read
    .format("parquet")
    .load(GOLD_DATE_PATH)
)

print("Gold customer dimension:", df_customer.count())
print("Gold product dimension:", df_product.count())
print("Gold date dimension:", df_date.count())

Gold customer dimension: 9997
Gold product dimension: 996
Gold date dimension: 731


### Inspect Source and Dimension Schemas

In [ ]:
print("TRANSACTION SCHEMA")
df_transaction.printSchema()

print("ORDER SCHEMA")
df_order.printSchema()

print("CUSTOMER DIMENSION SCHEMA")
df_customer.printSchema()

print("PRODUCT DIMENSION SCHEMA")
df_product.printSchema()

print("DATE DIMENSION SCHEMA")
df_date.printSchema()

TRANSACTION SCHEMA
root
 |-- transaction_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- discount_amount: decimal(10,2) (nullable = true)
 |-- tax_amount: decimal(10,2) (nullable = true)
 |-- transaction_amount: decimal(10,2) (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

ORDER SCHEMA
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- shipping_country: string (nullable = true)
 |-- shipping_city: string (nullable = true)
 |-- order_total: decimal(10,2) (nullable = true)
 |-- _silver_processed_ti

### Validate Dimension Join Keys

In [ ]:
customer_key_metrics = df_customer.agg(
    F.count("*").alias("rows"),
    F.countDistinct("customer_id").alias("distinct_keys")
).first()

product_key_metrics = df_product.agg(
    F.count("*").alias("rows"),
    F.countDistinct("product_id").alias("distinct_keys")
).first()

date_key_metrics = df_date.agg(
    F.count("*").alias("rows"),
    F.countDistinct("date_key").alias("distinct_keys")
).first()

assert customer_key_metrics["rows"] == customer_key_metrics["distinct_keys"]
assert product_key_metrics["rows"] == product_key_metrics["distinct_keys"]
assert date_key_metrics["rows"] == date_key_metrics["distinct_keys"]

assert df_customer.filter(F.col("customer_id").isNull()).count() == 0
assert df_product.filter(F.col("product_id").isNull()).count() == 0
assert df_date.filter(F.col("date_key").isNull()).count() == 0

print("All dimension join keys are unique and non-null.")

All dimension join keys are unique and non-null.


### Prepare DataFrames for Joining

In [ ]:
df_t = df_transaction.alias("t")

df_o = (
    df_order
    .select(
        "order_id",
        "customer_id",
        "order_date",
        "order_status"
    )
    .alias("o")
)

df_c = (
    df_customer
    .select("customer_id")
    .alias("c")
)

df_p = (
    df_product
    .select("product_id")
    .alias("p")
)

df_d = (
    df_date
    .select("full_date", "date_key")
    .alias("d")
)

### Join Transaction with Orders

In [ ]:
df_transaction_order = (
    df_t.join(
        df_o,
        F.col("t.order_id") == F.col("o.order_id"),
        "left"
    )
    .select(
        F.col("t.transaction_id"),
        F.col("t.order_id"),
        F.col("t.product_id"),
        F.col("t.transaction_date"),
        F.col("t.quantity"),
        F.col("t.unit_price"),
        F.col("t.discount_amount"),
        F.col("t.tax_amount"),
        F.col("t.transaction_amount"),
        F.col("t.payment_status"),
        F.col("t._silver_processed_timestamp"),

        F.col("o.customer_id"),
        F.col("o.order_date"),
        F.col("o.order_status")
    )
)

print("Transactions after order join:", df_transaction_order.count())
df_transaction_order.show(5, truncate=False)

Transactions after order join: 49990
+--------------+--------+----------+----------------+--------+----------+---------------+----------+------------------+--------------+---------------------------+-----------+----------+------------+
|transaction_id|order_id|product_id|transaction_date|quantity|unit_price|discount_amount|tax_amount|transaction_amount|payment_status|_silver_processed_timestamp|customer_id|order_date|order_status|
+--------------+--------+----------+----------------+--------+----------+---------------+----------+------------------+--------------+---------------------------+-----------+----------+------------+
|T00000001     |O0028299|P00169    |2024-04-04      |2       |82.22     |16.44          |11.84     |159.84            |Captured      |2026-10-27 17:22:20.70993  |C007895    |2024-04-04|Completed   |
|T00000002     |O0017225|P00332    |2024-04-07      |5       |28.76     |7.19           |10.93     |147.54            |Captured      |2026-10-27 17:22:20.70993  |C0092

### Join Customer Dimension

In [ ]:
df_transaction_customer = (
    df_transaction_order.alias("t")
    .join(
        df_c,
        F.col("t.customer_id") == F.col("c.customer_id"),
        "left"
    )
    .select(
        F.col("t.*")
    )
)

print("Transactions after customer join:", df_transaction_customer.count())

Transactions after customer join: 49990


### Join Product Dimension

In [ ]:
df_transaction_product = (
    df_transaction_customer.alias("t")
    .join(
        df_p,
        F.col("t.product_id") == F.col("p.product_id"),
        "left"
    )
    .select(
        F.col("t.*")
    )
)

print("Transactions after product join:", df_transaction_product.count())

Transactions after product join: 49990


### Derive Date Key and Join Date Dimension

In [ ]:
df_transaction_with_date_key = (
    df_transaction_product
    .withColumn(
        "date_key",
        F.date_format("transaction_date", "yyyyMMdd").cast("int")
    )
)

df_transaction_date = (
    df_transaction_with_date_key.alias("t")
    .join(
        df_d,
        F.col("t.date_key") == F.col("d.date_key"),
        "left"
    )
    .select(
        F.col("t.*")
    )
)

print("Transactions after date join:", df_transaction_date.count())

Transactions after date join: 49990


### Select Gold Fact Attributes

In [ ]:
df_gold_fact_sales = (
    df_transaction_date
    .select(
        "transaction_id",
        "order_id",
        "customer_id",
        "product_id",
        "date_key",
        "order_date",
        "order_status",
        "payment_status",
        "quantity",
        "unit_price",
        "discount_amount",
        "tax_amount",
        "transaction_amount",
        "_silver_processed_timestamp"
    )
)

df_gold_fact_sales.printSchema()
df_gold_fact_sales.show(10, truncate=False)

root
 |-- transaction_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- discount_amount: decimal(10,2) (nullable = true)
 |-- tax_amount: decimal(10,2) (nullable = true)
 |-- transaction_amount: decimal(10,2) (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = true)

+--------------+--------+-----------+----------+--------+----------+------------+--------------+--------+----------+---------------+----------+------------------+---------------------------+
|transaction_id|order_id|customer_id|product_id|date_key|order_date|order_status|payment_status|quantity|unit_price|discount_amount|tax_amount|transa

### Add Gold Audit Metadata

In [ ]:
df_gold_fact_sales = df_gold_fact_sales.withColumn(
    "_gold_processed_timestamp",
    F.current_timestamp()
)

df_gold_fact_sales.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- discount_amount: decimal(10,2) (nullable = true)
 |-- tax_amount: decimal(10,2) (nullable = true)
 |-- transaction_amount: decimal(10,2) (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = true)
 |-- _gold_processed_timestamp: timestamp (nullable = false)



### Validate Gold Fact Table

In [ ]:
fact_metrics = df_gold_fact_sales.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("transaction_id").alias("distinct_transactions"),

    F.sum(
        F.when(F.col("transaction_id").isNull(), 1).otherwise(0)
    ).alias("null_transaction_ids"),

    F.sum(
        F.when(F.col("customer_id").isNull(), 1).otherwise(0)
    ).alias("null_customer_ids"),

    F.sum(
        F.when(F.col("product_id").isNull(), 1).otherwise(0)
    ).alias("null_product_ids"),

    F.sum(
        F.when(F.col("date_key").isNull(), 1).otherwise(0)
    ).alias("null_date_keys"),

    F.sum(
        F.when(F.col("transaction_amount").isNull(), 1).otherwise(0)
    ).alias("null_transaction_amounts"),

    F.sum(
        F.when(F.col("_gold_processed_timestamp").isNull(), 1).otherwise(0)
    ).alias("null_gold_timestamps")
).first()

for metric, value in fact_metrics.asDict().items():
    print(f"{metric}: {value}")

assert fact_metrics["row_count"] == EXPECTED_TRANSACTION_COUNT
assert fact_metrics["distinct_transactions"] == fact_metrics["row_count"]
assert fact_metrics["null_transaction_ids"] == 0
assert fact_metrics["null_gold_timestamps"] == 0

print("Core fact table validation passed.")

row_count: 49990
distinct_transactions: 49990
null_transaction_ids: 0
null_customer_ids: 4
null_product_ids: 0
null_date_keys: 0
null_transaction_amounts: 0
null_gold_timestamps: 0
Core fact table validation passed.


### Relationship Quality Metrics

In [ ]:
unmatched_order_count = (
    df_transaction_order
    .filter(F.col("customer_id").isNull())
    .count()
)

unmatched_product_count = (
    df_transaction_product
    .join(
        df_product.select("product_id").alias("p"),
        df_transaction_product.product_id == F.col("p.product_id"),
        "left_anti"
    )
    .count()
)

unmatched_customer_count = (
    df_transaction_customer
    .join(
        df_customer.select("customer_id").alias("c"),
        df_transaction_customer.customer_id == F.col("c.customer_id"),
        "left_anti"
    )
    .count()
)

unmatched_date_count = (
    df_transaction_date
    .filter(F.col("date_key").isNull())
    .count()
)

print("Transactions without matching orders:", unmatched_order_count)
print("Transactions without matching products:", unmatched_product_count)
print("Transactions without matching customers:", unmatched_customer_count)
print("Transactions without matching dates:", unmatched_date_count)

Transactions without matching orders: 4
Transactions without matching products: 183
Transactions without matching customers: 16
Transactions without matching dates: 0


### Write Gold Fact Table

In [ ]:
(
    df_gold_fact_sales
    .write
    .format("parquet")
    .mode("overwrite")
    .save(GOLD_FACT_SALES_PATH)
)

print("Gold fact table saved to:", GOLD_FACT_SALES_PATH)

Gold fact table saved to: /content/drive/MyDrive/pyspark-ecommerce-analytics-data/gold/fact_sales


### Read Back and Verify

In [ ]:
df_gold_fact_sales_check = (
    spark.read
    .format("parquet")
    .load(GOLD_FACT_SALES_PATH)
)

saved_fact_count = df_gold_fact_sales_check.count()

print("Expected transaction count:", EXPECTED_TRANSACTION_COUNT)
print("Saved fact table count:", saved_fact_count)

df_gold_fact_sales_check.printSchema()
df_gold_fact_sales_check.show(10, truncate=False)

assert saved_fact_count == EXPECTED_TRANSACTION_COUNT
assert (
    df_gold_fact_sales_check.select("transaction_id").distinct().count()
    == saved_fact_count
)

print("Gold fact read-back verification passed.")

Expected transaction count: 49990
Saved fact table count: 49990
root
 |-- transaction_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- discount_amount: decimal(10,2) (nullable = true)
 |-- tax_amount: decimal(10,2) (nullable = true)
 |-- transaction_amount: decimal(10,2) (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = true)
 |-- _gold_processed_timestamp: timestamp (nullable = true)

+--------------+--------+-----------+----------+--------+----------+------------+--------------+--------+----------+---------------+----------+------------------+---------------------------+--------------------------